# Avocado Price Prediction - Data Cleaning and Feature Engineering

Pipeline:

1. Drop leftover unnamed index columns
2. Pase `date` -> extract month , week 
3. Rename PLU columns to readable names
4. Eng `log_total_volume, log_total_bag, small_share, large_share, xl_share`
5. One-Hot-Encode `type` and `region`
6. Save to a new file

## 1. Import & Load

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sys

sns.set_style('whitegrid')
from utils import *

%matplotlib inline

In [2]:
df= pd.read_csv(r'data\avocado.csv')
df

,Unnamed: 0,Date,AveragePrice,Total Volume,4046,4225,4770,Total Bags,Small Bags,Large Bags,XLarge Bags,type,year,region
0,0,2015-12-27,1.33,64236.62,1036.74,54454.85,48.16,8696.87,8603.62,93.25,0.0,conventional,2015,Albany
1,1,2015-12-20,1.35,54876.98,674.28,44638.81,58.33,9505.56,9408.07,97.49,0.0,conventional,2015,Albany
2,2,2015-12-13,0.93,118220.22,794.70,109149.67,130.50,8145.35,8042.21,103.14,0.0,conventional,2015,Albany
3,3,2015-12-06,1.08,78992.15,1132.00,71976.41,72.58,5811.16,5677.40,133.76,0.0,conventional,2015,Albany
4,4,2015-11-29,1.28,51039.60,941.48,43838.39,75.78,6183.95,5986.26,197.69,0.0,conventional,2015,Albany
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18244,7,2018-02-04,1.63,17074.83,2046.96,1529.20,0.00,13498.67,13066.82,431.85,0.0,organic,2018,WestTexNewMexico
18245,8,2018-01-28,1.71,13888.04,1191.70,3431.50,0.00,9264.84,8940.04,324.80,0.0,organic,2018,WestTexNewMexico
18246,9,2018-01-21,1.87,13766.76,1191.92,2452.79,727.94,9394.11,9351.80,42.31,0.0,organic,2018,WestTexNewMexico
18247,10,2018-01-14,1.93,16205.22,1527.63,2981.04,727.01,10969.54,10919.54,50.00,0.0,organic,2018,WestTexNewMexico


## 2. Missing / Invalid Input

In [3]:
print(f"Missing Values: {df.isnull().sum().sum()}")
print(f"Duplicated Rows: {df.duplicated().sum()}")

Missing Values: 0
Duplicated Rows: 0


## 3. Drop index, Parse Date, Rename PLU columns

In [4]:
df_step= drop_index(df)
df_step= parse_data(df_step)
df_step = rename_plu(df_step)
print(f"New Column: {[x for x in df_step.columns]}")
df_step.head()

New Column: ['Date', 'AveragePrice', 'total_volume', 'small_hass', 'large_hass', 'xl_hass', 'total_bags', 'small_bags', 'large_bags', 'xl_bags', 'type', 'year', 'region', 'Month', 'Week']


,Date,AveragePrice,total_volume,small_hass,large_hass,xl_hass,total_bags,small_bags,large_bags,xl_bags,type,year,region,Month,Week
0,2015-12-27,1.33,64236.62,1036.74,54454.85,48.16,8696.87,8603.62,93.25,0.0,conventional,2015,Albany,12,52
1,2015-12-20,1.35,54876.98,674.28,44638.81,58.33,9505.56,9408.07,97.49,0.0,conventional,2015,Albany,12,51
2,2015-12-13,0.93,118220.22,794.70,109149.67,130.50,8145.35,8042.21,103.14,0.0,conventional,2015,Albany,12,50
3,2015-12-06,1.08,78992.15,1132.00,71976.41,72.58,5811.16,5677.40,133.76,0.0,conventional,2015,Albany,12,49
4,2015-11-29,1.28,51039.60,941.48,43838.39,75.78,6183.95,5986.26,197.69,0.0,conventional,2015,Albany,11,48


## 4. Feature Engineering

In [5]:
df_feat = create_feature(df_step)

new_col = [c for c in df_feat.columns if c not in df_step.columns]
print(f"New Columns: {new_col}")

df_feat[new_col + ['AveragePrice']].head()

New Columns: ['log_total_volume', 'log_total_bags', 'log_small_bags', 'log_large_bags', 'log_xl_bags', 'small_share', 'large_share', 'xl_share']


,log_total_volume,log_total_bags,log_small_bags,log_large_bags,log_xl_bags,small_share,large_share,xl_share,AveragePrice
0,11.070344,9.070833,9.060055,4.545951,0.0,0.018667,0.980466,0.000867,1.33
1,10.912867,9.159737,9.149429,4.589955,0.0,0.014861,0.983853,0.001286,1.35
2,11.680313,9.005325,8.992584,4.645736,0.0,0.007220,0.991595,0.001186,0.93
3,11.277116,8.667708,8.644425,4.903495,0.0,0.015468,0.983540,0.000992,1.08
4,10.840377,8.729874,8.697389,5.291746,0.0,0.020989,0.977321,0.001689,1.28
